In [1]:
from pathlib import Path
import json
import sys
import pandas as pd

candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / "work",
              Path("/Users/quanh.hg/python/work")]
WORK_DIR = next((p.resolve() for p in candidates
                 if (p / "data/hanoi_tiles/project_layout.json").is_file()
                 and (p / "scripts/overpass/download_tiles.py").is_file()), None)
if WORK_DIR is None:
    raise FileNotFoundError("Không tìm thấy dự án: cần data/hanoi_tiles và scripts/overpass trong work.")
ROOT = WORK_DIR / "data" / "hanoi_tiles"
SOURCE_DIR = WORK_DIR / "scripts" / "overpass"
loaded = sys.modules.get("download_tiles")
if loaded is not None and (Path(loaded.__file__).resolve().parent != SOURCE_DIR
                           or not hasattr(loaded, "project_file")
                           or Path(loaded.ROOT).resolve() != ROOT):
    raise RuntimeError("Đường dẫn dự án đã đổi. Restart Kernel rồi chạy lại notebook.")
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))
from project_paths import project_file
tiles = pd.read_csv(ROOT / "tiles.csv", keep_default_na=False)
print("Số ô đang hoạt động:", len(tiles))
tiles[["tile_id", "side_m", "scope_fraction", "status"]].head(10)


In [2]:
from IPython.display import Image, display

qa = json.loads((project_file(ROOT, "qa_report.json")).read_text())
print("Kiểm tra đạt:", qa["passed"])
print("Diện tích ranh OSM (km²):", round(qa["boundary_area_km2"], 2))
print("Diện tích ranh chưa được phủ (m²):", qa["uncovered_scope_area_m2"])
print("Diện tích ô vuông chồng nhau (m²):", qa["projected_square_overlap_area_m2"])
display(Image(filename=str(project_file(ROOT, "hanoi_download_grid.png")), width=800))


In [3]:
TILE_ID = "HN_R011_C012"
selected = tiles.loc[tiles["tile_id"] == TILE_ID]
if len(selected) != 1:
    raise ValueError("Mã ô không có trong bảng hiện tại; hãy chọn một ô lá.")
tile = selected.iloc[0]
BBOX = tuple(float(tile[k]) for k in ("south", "west", "north", "east"))
SNAPSHOT_UTC = str(tile["snapshot_utc"])
QUERY = (ROOT / str(tile["query_file"])).read_text(encoding="utf-8")
config = json.loads((ROOT / "tile_project.json").read_text())
ENDPOINT = config["endpoint"]

print("TILE_ID =", repr(TILE_ID))
print("BBOX =", repr(BBOX))
print("SNAPSHOT_UTC =", repr(SNAPSHOT_UTC))
print("Endpoint:", ENDPOINT)
print("\nTruy vấn:\n" + QUERY)

### Hình học: tùy chọn, cần kernel PyQGIS

Phần tải HTTP bên dưới dùng Python thường và requests. Chỉ bật kiểm tra ranh giới khi dùng kernel PyQGIS. Không chia ô khi bộ tải đang chạy.


In [4]:
ENABLE_PYQGIS = False
if ENABLE_PYQGIS:
    sys.path.insert(0, str(SOURCE_DIR))
    import build_tiles as grid
    report = grid.verify(ROOT)
    print(json.dumps(report, ensure_ascii=False, indent=2))
else:
    print("Đã bỏ qua kiểm tra hình học. Bộ tải HTTP không cần QGIS.")


In [5]:
def split_heavy_tile(tile_id):
    # Chỉ gọi bằng kernel PyQGIS, sau khi dừng bộ tải.
    sys.path.insert(0, str(SOURCE_DIR))
    import build_tiles as grid
    result = grid.split_tile(tile_id, root=ROOT)
    print("Ô con:", result["children"])
    print("Độ phủ sau chia đạt:", result["qa"]["passed"])
    return pd.read_csv(ROOT / "tiles.csv", keep_default_na=False)

# Bộ tải HTTP không gọi hàm chia ô này.


### 1. Cấu hình bộ tải

Endpoint, snapshot, query và thư mục kết quả được đọc từ các tệp của dự án. Bộ tải dùng một Session, gửi tuần tự và kiểm tra cache trước từng ô. max_attempts là tổng số lần thử tối đa cho mỗi ô trong một lượt chạy.


In [6]:
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))
import download_tiles as downloader

OPTIONS = downloader.DownloadOptions(
    interval_seconds=15,
    connect_timeout=10,
    read_timeout=240,
    max_attempts=3,
    backoff_base_seconds=30,
    backoff_max_seconds=300,
    jitter_seconds=5,
    retry_failed=True,
    retry_heavy=True,
)


### 2. Kiểm tra cấu hình và xem tiến độ

Cell này không gửi API. Nó kiểm tra mã ô, query, snapshot, đường dẫn và đối chiếu CSV với lịch sử. Các ô split được bỏ qua.


In [7]:
check = downloader.preflight(ROOT, OPTIONS)
print(json.dumps(check, ensure_ascii=False, indent=2))
status = downloader.status_report(ROOT)
display(pd.DataFrame([status["counts"]]))


### 3. Chạy tải tất cả các ô đang hoạt động

Cell này gửi request HTTP. Cache hợp lệ được đọc lại; các ô cần tải được xử lý tuần tự. Ctrl+C/Interrupt dừng lượt chạy và giữ kết quả đã hoàn thành. Không chạy thêm bộ tải trong terminal hoặc notebook khác cùng lúc.


In [8]:
run_report = downloader.run_download(ROOT, options=OPTIONS)
display(pd.DataFrame([{
    **run_report["counts"],
    "tải_mới": run_report["new_downloads"],
    "đọc_cache": run_report["cache_hits"],
    "thời_gian_giây": run_report["elapsed_seconds"],
    "đã_dừng": run_report["interrupted"],
}]))


### 4. Tiếp tục sau khi dừng

Chạy lại cell tải phía trên là đủ để tiếp tục. Hoặc đổi RESUME_NOW=True ở cell dưới và chạy riêng cell đó. Bộ tải kiểm tra lại hash, query/snapshot và phản hồi trước khi bỏ qua cache; không dùng mỗi chữ done làm bằng chứng.


In [9]:
RESUME_NOW = False
if RESUME_NOW:
    run_report = downloader.run_download(ROOT, options=OPTIONS)
    print(json.dumps(run_report, ensure_ascii=False, indent=2))
else:
    print("Để tiếp tục: chạy lại cell tải, hoặc đặt RESUME_NOW=True và chạy cell này.")


### 5. Ô thất bại và báo cáo tổng kết

heavy là ô vẫn gặp timeout/quá tải sau số lần thử giới hạn; chưa tự chia ô. failed cần xem nguyên nhân. Nhật ký từng ô nằm trong response_dir/errors.jsonl; báo cáo gần nhất là latest_run.json. CSV và lịch sử được cập nhật sau từng ô.


In [10]:
status = downloader.status_report(ROOT)
tiles = pd.read_csv(ROOT / "tiles.csv", keep_default_na=False)
display(pd.DataFrame([status["counts"]]))
problems = pd.DataFrame(status["problems"], columns=[
    "tile_id", "status", "last_error_kind", "last_error", "download_attempts"
])
display(problems)
latest = status["latest_run"]
if latest:
    display(pd.DataFrame([{
        "lượt_chạy": latest["run_id"],
        "tải_mới": latest["new_downloads"],
        "đọc_cache": latest["cache_hits"],
        "request_HTTP": latest["http_requests"],
        "ô_không_có_đường": latest["empty_road_results"],
        "thời_gian_giây": latest["elapsed_seconds"],
        "đã_dừng": latest["interrupted"],
    }]))
